In [3]:
import pandas as pd
import numpy as np
import random 

# Set a seed for reproducibility
np.random.seed(0)

# Generating a list of unique counterparty_IDs
counterparty_ids = range(1, 1001)

# Generating a list of industries 
industries = ['Banking and Financial Services', 'Manufacturing', 'Services', 'Retail', 'Technology', 'Energy and Utilities', 'Real Estate and Construction']

# Generating a lsit of countries 
countries = ['US', 'UK', 'Germany', 'France', 'Switzerland', 'China', 'South Korea']

# Generating a list of ratings 
ratings = ['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'CCC', 'CC', 'C', 'D']


# Generate random EAD, PD, and LGD values
ead = np.random.uniform(low = 1e6, high = 1e9, size = 1000)  # Exposure at defult
probability_of_default = np.random.uniform(low = 0.01, high = 0.2, size = 1000)  # Probability of default
lgd = np.random.uniform(low = 0.2, high = 0.8, size = 1000)  # Loss given default

# Generate random industries, countries, and ratings
industry = [random.choice(industries) for _ in counterparty_ids]
country = [random.choice(countries) for _ in counterparty_ids]
rating = [random.choice(ratings) for _ in counterparty_ids]

# Generate a DataFrame 
data = pd.DataFrame({
    'CounterpartyID': counterparty_ids,
    'EAD': ead,
    'PD': probability_of_default,
    'LGD': lgd, 'Industry': industry,
    'Country': country,
    'Rating': rating
})
data.head()

,CounterpartyID,EAD,PD,LGD,Industry,Country,Rating
0,1,5.492647e+08,0.122647,0.686911,Manufacturing,France,AAA
1,2,7.154742e+08,0.011912,0.485650,Retail,South Korea,C
2,3,6.031606e+08,0.100407,0.513894,Retail,US,A
3,4,5.453383e+08,0.144666,0.350312,Services,China,CC
4,5,4.242311e+08,0.018355,0.563026,Retail,Germany,BB


In [ ]:
# Calculate the Capital Requirement under Pillar 1
data['Pillar_I_Capital_Requirement'] = data['EAD'] * data['PD'] * data['LGD'] *12.5

# Calculate the stressed capital requirement for CCAR
data['Stressed_PD'] = data['PD'] * 1.5     # 50% increase in PD
data['Stressed_LGD'] = data['LGD'] * 1.2   # 20% increase in LGD
data['Capital_Requirement_CCAR'] = data['EAD'] * data['Stressed_PD'] * data['Stressed_LGD'] * 12.5

# Calculate the expected credit loss for IFRS-9
data['Expected_Credit_Loss_IFRS9'] = data['EAD'] * data['PD'] * data['LGD']

# Calculate the stressed capital requirement for internal stress testing
data['Stressed_PD_Internal'] = data['PD'] * 2.0  # 100% increase in PD
data['Stressed_LGD_Internal'] = data['LGD'] * 1.5  # 50% increase in LGD
data['Capital_Requirement_Internal_Stress_Test'] = data['EAD'] * data['Stressed_PD_Internal'] * data['Stressed_LGD_Internal'] * 12.5

data['Perc_Increase_Baseline_to_Severe'] = (data['Capital_Requirement_Internal_Stress_Test'] - data['Pillar_I_Capital_Requirement']) / data['Pillar_I_Capital_Requirement'] * 100

# severity matrics
max_increase = data['Perc_Increase_Baseline_to_Severe'].max()
print(f"Maximum capital increase under severe stress: {max_increase:.2f}%")
